In [1]:
import warnings
from pathlib import Path

from statsmodels.tools.sm_exceptions import InterpolationWarning

from input import input
import config
from model import generics, hybrid_system_exp, grid_search_exp
from model.feature_selection import TimeSeriesFeatureSelector
from metrics import metrics
import numpy as np

from sklearn.neural_network import MLPRegressor
from sklearn.pipeline import Pipeline
import pandas as pd

%load_ext autoreload
%autoreload 2

Failed to read module file 'C:\Users\joaol\AppData\Local\Programs\Python\Python311\Lib\functools.py' for module 'functools': UnicodeDecodeError
Traceback (most recent call last):
  File "c:\Projetos\mestrado_codigos\experiments\.venv\Lib\site-packages\IPython\core\extensions.py", line 62, in load_extension
    return self._load_extension(module_str)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Projetos\mestrado_codigos\experiments\.venv\Lib\site-packages\IPython\core\extensions.py", line 77, in _load_extension
    mod = import_module(module_str)
          ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\joaol\AppData\Local\Programs\Python\Python311\Lib\importlib\__init__.py", line 126, in import_module
    return _bootstrap._gcd_import(name[level:], package, level)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "<frozen importlib._bootstrap>", line 1204, in _gcd_import
  File "<frozen importlib._bootstrap>", line 1176, in _find_and_load
  File "<frozen impor

In [2]:
# === CELULA DE CONFIGURACAO -- Tarefa 3.2 (fluxo notebook-only) ===
# Edite aqui para uma nova rodada: series, experiment_id, grid. Ver
# RUNBOOK.md Secao 1 (series/lag_size) e Secao 3 (convencao de experiment_id).
#
# Tarefa 3 do PLANO_ARQUITETURA.md: integracao do TimeSeriesFeatureSelector
# via sklearn.Pipeline. strategy fixa nesta execucao ('mutual_info' -- filtro
# por teoria da informacao, promove o experimento exploratorio ja existente
# em src/test_mutual_information.py para a arquitetura formal). k varia como
# grid (selector__k abaixo). A transformacao de y (MinMaxScaler/KPSS) continua
# fora do Pipeline, dentro de Additive/generics.format_forecats (Secao 1.4 do
# PLANO_ARQUITETURA.md).
model = Pipeline([
    ('selector', TimeSeriesFeatureSelector(strategy='mutual_info')),
    ('estimator', MLPRegressor(activation='logistic', solver='lbfgs')),
])

# Series a rodar nesta execucao (FS_DEV_SERIES por padrao -- tests/model/conftest.py).
# austres.txt resolve para 1 unico lag via lag_size='auto' (RUNBOOK.md Secao 1) --
# avaliar se faz sentido incluir antes de comparar metodos de FS entre si.
fs_series_list = ['airlines.txt', 'austres.txt', 'coloradoRiver.txt', 'sunspot.txt']

# experiment_id novo e explicito (Secao 3.2 do CLAUDE.md) -- nunca reaproveitar.
experiment_id = 'chamados_v4_fs_mutualinfo'
# Caminhos derivados de experiment_id, computados uma vez aqui e reusados nas
# celulas 3/5/6 (achado de code-review da Tarefa 3.2 -- evita reconstruir o
# mesmo Path 3x).
experiment_dir = Path(config.MODEL_DATA_PATH) / experiment_id
experiment_dir_results = Path(config.ROOT_PATH) / 'results' / experiment_id

# Sufixo sem underscore -- extract_series_name_from_path/model_name em
# metrics.py continuam funcionando sem nenhuma mudanca (ver Tarefa 3, Parte B).
model_name = 'amv1mutualinfo'
normalize = True
force = False
model_exec = 10

experiment_params = {
    'linear_model_name': '1arima',
    'diff_kpss': False,
    'horizon': 1,
}

# Convencao selector__*/estimator__* (nativa do sklearn): GridSearch usa
# clone(model).set_params(**params), que resolve essas chaves automaticamente
# via Pipeline.get_params(deep=True) -- nenhuma mudanca em grid_search_exp.py.
model_parameters = {
    'selector__k': [1, 5, 9, 15, 20],
    'estimator__hidden_layer_sizes': [10, 20, 50],
    'estimator__max_iter': [1000],
   }

In [3]:
# Sanity-check exigido pela Tarefa 2: confirma que Pipeline.get_params(deep=True)
# expoe as chaves selector__*/estimator__* que GridSearch (ParameterGrid +
# clone().set_params()) vai usar, antes de rodar o grid completo.
params = model.get_params(deep=True)
required_keys = {'selector__strategy', 'selector__k', 'estimator__hidden_layer_sizes', 'estimator__max_iter'}
missing = required_keys - params.keys()
assert not missing, f"get_params(deep=True) nao expos chaves esperadas: {missing}"
print("OK -- Pipeline.get_params(deep=True) expoe:", sorted(required_keys))

# Checagem de identidade (achado de code-review da Tarefa 3.2): trava que a
# 'strategy' do seletor (celula 1) e o experiment_id/model_name declarados
# na MESMA celula continuam consistentes entre si -- protege contra editar
# 'strategy' na celula de configuracao por engano e deixar experiment_id/
# model_name (e portanto o .pkl/CSV gerado) mislabeled com o nome de outra
# estrategia.
strategy_slug = model.named_steps['selector'].strategy.replace('_', '')
assert strategy_slug in experiment_id, (
    f"strategy={model.named_steps['selector'].strategy!r} nao aparece em "
    f"experiment_id={experiment_id!r} -- confirme que voce nao mudou 'strategy' "
    "na celula de configuracao sem atualizar experiment_id/model_name."
)
print(f"OK -- strategy={model.named_steps['selector'].strategy!r} consistente com experiment_id={experiment_id!r}")

OK -- Pipeline.get_params(deep=True) expoe: ['estimator__hidden_layer_sizes', 'estimator__max_iter', 'selector__k', 'selector__strategy']
OK -- strategy='mutual_info' consistente com experiment_id='chamados_v4_fs_mutualinfo'


In [4]:
# Copia o ARIMA pre-treinado (mesmo modelo, nao retreina) para o novo
# experiment_id -- Additive precisa dele sob o MESMO experiment_id da variante
# FS (ver RUNBOOK.md Secao 3). Usa copy_pretrained_linear_model (src/utils/),
# que por baixo usa generics.format_names() -- o mesmo helper que Additive/
# input_linear_info usam para localizar esse .pkl -- e le o nome do modelo
# linear de experiment_params['linear_model_name'] (celula 1), em vez de um
# literal cravado (achado de code-review da Tarefa 3.2).
from utils.copy_pretrained_linear_model import copy_pretrained_linear_model

copy_pretrained_linear_model(
    source_experiment_id='chamados',
    dest_experiment_id=experiment_id,
    series_list=fs_series_list,
    linear_model_name=experiment_params['linear_model_name'],
)

airlines: C:\Projetos\mestrado_codigos\experiments\data\result\chamados\airlines_1arima.pkl -> C:\Projetos\mestrado_codigos\experiments/data/result/chamados_v4_fs_mutualinfo/airlines_1arima.pkl
austres: C:\Projetos\mestrado_codigos\experiments\data\result\chamados\austres_1arima.pkl -> C:\Projetos\mestrado_codigos\experiments/data/result/chamados_v4_fs_mutualinfo/austres_1arima.pkl
coloradoRiver: C:\Projetos\mestrado_codigos\experiments\data\result\chamados\coloradoRiver_1arima.pkl -> C:\Projetos\mestrado_codigos\experiments/data/result/chamados_v4_fs_mutualinfo/coloradoRiver_1arima.pkl
sunspot: C:\Projetos\mestrado_codigos\experiments\data\result\chamados\sunspot_1arima.pkl -> C:\Projetos\mestrado_codigos\experiments/data/result/chamados_v4_fs_mutualinfo/sunspot_1arima.pkl


In [5]:
# Chamado direto via GridSearch(...).execution() por serie, em vez de
# grid_seach_multiple_bases(), para nao depender/mutar config.BASE_NAME_LIST e
# sem tocar grid_search_exp.py. use_val_slipt_for_prev=True e explicito porque
# o default de GridSearch (False) diverge do default do wrapper
# grid_seach_multiple_bases (True) -- sem isso, o refit final perderia o
# val_size e os .pkl ficariam sem val_metrics, quebrando a paridade com o
# baseline original (amv1 em chamados/).
for base_name in fs_series_list:
    print(base_name)
    exec_gs = grid_search_exp.GridSearch(
        hybrid_system_exp.Additive,
        model,
        model_parameters,
        experiment_id,
        base_name,
        model_name,
        force,
        normalize,
        experiment_params,
        model_exec=model_exec,
        use_val_slipt_for_prev=True,
    )
    exec_gs.execution()

airlines.txt
{'estimator__hidden_layer_sizes': 50, 'estimator__max_iter': 1000, 'selector__k': 9}
austres.txt
{'estimator__hidden_layer_sizes': 50, 'estimator__max_iter': 1000, 'selector__k': 5}
coloradoRiver.txt
{'estimator__hidden_layer_sizes': 10, 'estimator__max_iter': 1000, 'selector__k': 15}
sunspot.txt
{'estimator__hidden_layer_sizes': 50, 'estimator__max_iter': 1000, 'selector__k': 20}


In [6]:
# Gera o CSV de metricas (agregado + detalhado) direto no notebook -- mesma
# funcao usada pela CLI (src/utils/export_metrics_to_csv.py), so o ponto de
# entrada muda (Tarefa 3.2, fluxo notebook-only). Equivale ao Passo 2 do
# RUNBOOK.md Secao 8b.
from utils.export_metrics_to_csv import run_export_metrics_to_csv

metrics_output = experiment_dir_results / 'metrics.csv'
df_metrics = run_export_metrics_to_csv(
    experiment_dir,
    metrics_output,
    detail=True,
)
df_metrics

[INFO] 8 arquivo(s) .pkl encontrado(s) em 'C:\Projetos\mestrado_codigos\experiments\data\result\chamados_v4_fs_mutualinfo'.

  OK  airlines_1amv1mutualinfo.pkl  ->  10 linha(s)
  OK  airlines_1arima.pkl  ->  1 linha(s)
  OK  austres_1amv1mutualinfo.pkl  ->  10 linha(s)
  OK  austres_1arima.pkl  ->  1 linha(s)
  OK  coloradoRiver_1amv1mutualinfo.pkl  ->  10 linha(s)
  OK  coloradoRiver_1arima.pkl  ->  1 linha(s)
  OK  sunspot_1amv1mutualinfo.pkl  ->  10 linha(s)
  OK  sunspot_1arima.pkl  ->  1 linha(s)

[OK] CSV agregado (média das repetições) gerado em: C:\Projetos\mestrado_codigos\experiments\results\chamados_v4_fs_mutualinfo\metrics.csv
     8 linha(s) × 20 coluna(s)

[OK] CSV detalhado (por repetição) gerado em: C:\Projetos\mestrado_codigos\experiments\results\chamados_v4_fs_mutualinfo\metrics_detail.csv
     44 linha(s) × 13 coluna(s)


,ExperimentID,Serie,Modelo,N_Repeticoes,MSE_mean,MSE_std,RMSE_mean,RMSE_std,MAE_mean,MAE_std,MAPE_mean,MAPE_std,theil_mean,theil_std,ARV_mean,ARV_std,IA_mean,IA_std,POCID_mean,POCID_std
0,chamados_v4_fs_mutualinfo,airlines,1amv1mutualinfo,10,384.717102,2.482601,19.614115,0.063294,15.489698,0.075691,3.450297,0.016561,0.145776,0.000786,0.066363,0.000443,0.983230,0.000111,78.571429,0.000000
1,chamados_v4_fs_mutualinfo,airlines,1arima,1,389.194049,NaN,19.728002,NaN,15.102607,NaN,3.315326,NaN,0.134713,NaN,0.066301,NaN,0.983139,NaN,78.571429,NaN
2,chamados_v4_fs_mutualinfo,austres,1amv1mutualinfo,10,362.357858,12.615030,19.033145,0.328701,13.956760,0.369683,0.079794,0.002102,0.131166,0.003450,0.035296,0.001106,0.990894,0.000304,75.000000,0.000000
3,chamados_v4_fs_mutualinfo,austres,1arima,1,358.130880,NaN,18.924346,NaN,13.710292,NaN,0.078380,NaN,0.129713,NaN,0.034904,NaN,0.990999,NaN,75.000000,NaN
4,chamados_v4_fs_mutualinfo,coloradoRiver,1amv1mutualinfo,10,0.108262,0.003085,0.329002,0.004690,0.265919,0.003623,29.165356,0.466152,0.761637,0.020226,0.702912,0.006926,0.662601,0.004922,54.189189,2.421564
5,chamados_v4_fs_mutualinfo,coloradoRiver,1arima,1,0.104961,NaN,0.323976,NaN,0.262984,NaN,28.817193,NaN,0.755655,NaN,0.698832,NaN,0.667249,NaN,52.702703,NaN
6,chamados_v4_fs_mutualinfo,sunspot,1amv1mutualinfo,10,364.510928,2.426371,19.092074,0.063498,15.792911,0.145264,40.331124,0.501832,0.336788,0.002497,0.203949,0.001789,0.950065,0.000335,60.714286,0.000000
7,chamados_v4_fs_mutualinfo,sunspot,1arima,1,365.478709,NaN,19.117497,NaN,15.719171,NaN,36.503112,NaN,0.346894,NaN,0.193887,NaN,0.951367,NaN,67.857143,NaN


In [7]:
# Gera o CSV de features selecionadas (agregado + detalhado) direto no
# notebook -- mesma funcao usada pela CLI (src/utils/export_selected_features.py),
# so o ponto de entrada muda (Tarefa 3.2, fluxo notebook-only). Equivale ao
# Passo 2 do RUNBOOK.md Secao 8b.
from utils.export_selected_features import run_export_selected_features

features_output = experiment_dir_results / 'selected_features.csv'
df_features = run_export_selected_features(
    experiment_dir,
    features_output,
    detail=True,
)
df_features

Failed to read module file 'C:\Projetos\mestrado_codigos\experiments\src\utils\export_metrics_to_csv.py' for module 'utils.export_metrics_to_csv': UnicodeDecodeError
Traceback (most recent call last):
  File "c:\Projetos\mestrado_codigos\experiments\.venv\Lib\site-packages\IPython\extensions\deduperreload\deduperreload.py", line 219, in update_sources
    self.source_by_modname[new_modname] = f.read()
                                          ^^^^^^^^
  File "C:\Users\joaol\AppData\Local\Programs\Python\Python311\Lib\encodings\cp1252.py", line 23, in decode
    return codecs.charmap_decode(input,self.errors,decoding_table)[0]
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
UnicodeDecodeError: 'charmap' codec can't decode byte 0x90 in position 3950: character maps to <undefined>


[INFO] 8 arquivo(s) .pkl encontrado(s) em 'C:\Projetos\mestrado_codigos\experiments\data\result\chamados_v4_fs_mutualinfo'.

  OK  airlines_1amv1mutualinfo.pkl  ->  10 linha(s)
  OK  airlines_1arima.pkl  ->  sem seletor -- ignorado
  OK  austres_1amv1mutualinfo.pkl  ->  10 linha(s)
  OK  austres_1arima.pkl  ->  sem seletor -- ignorado
  OK  coloradoRiver_1amv1mutualinfo.pkl  ->  10 linha(s)
  OK  coloradoRiver_1arima.pkl  ->  sem seletor -- ignorado
  OK  sunspot_1amv1mutualinfo.pkl  ->  10 linha(s)
  OK  sunspot_1arima.pkl  ->  sem seletor -- ignorado

[OK] CSV agregado (média/desvio por série × modelo) gerado em: C:\Projetos\mestrado_codigos\experiments\results\chamados_v4_fs_mutualinfo\selected_features.csv
     4 linha(s) × 8 coluna(s)

[OK] CSV detalhado (por repetição) gerado em: C:\Projetos\mestrado_codigos\experiments\results\chamados_v4_fs_mutualinfo\selected_features_detail.csv
     40 linha(s) × 9 coluna(s)


,ExperimentID,Serie,Modelo,Strategy,N_Features_Selected_mean,N_Features_Selected_std,N_Repeticoes,N_Features_Total
0,chamados_v4_fs_mutualinfo,airlines,1amv1mutualinfo,mutual_info,9.0,0.0,10,20
1,chamados_v4_fs_mutualinfo,austres,1amv1mutualinfo,mutual_info,1.0,0.0,10,1
2,chamados_v4_fs_mutualinfo,coloradoRiver,1amv1mutualinfo,mutual_info,15.0,0.0,10,16
3,chamados_v4_fs_mutualinfo,sunspot,1amv1mutualinfo,mutual_info,9.0,0.0,10,9
